In [ ]:
import pandas as pd
from tqdm import tqdm
tqdm.pandas()
from dotenv import load_dotenv
import os
from sklearn.metrics.pairwise import cosine_similarity
from huggingface_hub import InferenceClient
import numpy as np
import os

In [ ]:
from sentence_transformers import SentenceTransformer, util


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#file_path = os.path.join(os.path.dirname(__file__),'/content/drive/MyDrive/Hackathon/state_scheme.pkl')
file_path = '/content/drive/MyDrive/Hackathon/state_scheme.pkl'
if os.path.exists(file_path):
    df = pd.read_pickle(file_path)
else:
    raise FileNotFoundError(f"File not found at {file_path}")


In [ ]:
df.shape

(2859, 10)

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
text="i am  poor farmer"
sentence_emb = model.encode([text], convert_to_tensor=True)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import tensorflow as tf

def compare(sentence_emb,scheme_embedding):
  sentence_embedding_2d = sentence_emb.numpy().reshape(1, -1)
  scheme_embedding_2d = scheme_embedding.numpy().reshape(1, -1)
  # Calculate the cosine similarity
  similarity = cosine_similarity(sentence_embedding_2d, scheme_embedding_2d)
  return similarity.item()

In [ ]:
df.head()

,scheme_name,details,benefits,documents,schemeCategory,tags,state,tokens,embeddings,scheme_id
0,"""Immediate Relief Assistance"" under ""Welfare a...","The scheme ""Immediate Relief Assistance"" is a ...","₹ 1,00,000, in two installments of ₹ 50,000 ea...",Photograph of the Family (Legal Heir) of the M...,"Agriculture,Rural & Environment, Social welfar...","Missing, Fisherman, Relief, Financial Assistan...",Puducherry,"The scheme ""Immediate Relief Assistance"" is a ...","[tensor(-0.0654), tensor(0.0842), tensor(-0.04...",S0000
1,Burial and Ex-gratia Payment Scheme in Case of...,"Launched in 2014, the "" Burial and Ex-gratia P...","Funeral Assistance: ₹3,000 payable in case of ...",Aadhaar Card of the applicant (nominee/Legal h...,Social welfare & Empowerment,"Building Worker, Construction Workers, Unregis...",Madhya Pradesh,"Launched in 2014, the "" Burial and Ex-gratia P...","[tensor(-0.0749), tensor(0.0051), tensor(-0.08...",S0001
2,Garuda Scheme for Funeral Expense,Andhra Pradesh Brahmin Welfare Corporation (AB...,"Financial Assistance of ₹10,000/- for funeral ...",Passport-size Photograph of the Applicant Aadh...,Social welfare & Empowerment,"Social Welfare, Financial Assistance, Deceased...",Andhra Pradesh,Andhra Pradesh Brahmin Welfare Corporation (AB...,"[tensor(-0.0639), tensor(0.0410), tensor(-0.08...",S0002
3,Incentive For The Intra Caste Marriage within ...,"The scheme ""Incentive for Intra Caste Marriage...","An incentive of ₹2,00,000/- is provided to the...",Aadhaar Card Bank Account Details Marriage Pro...,Social welfare & Empowerment,"Intra Caste, Marriage, Schedule Tribe, Incenti...",Karnataka,"The scheme ""Incentive for Intra Caste Marriage...","[tensor(-0.0722), tensor(0.0826), tensor(-0.09...",S0003
4,Incentive Scheme for MSMEs in Powerloom Sector...,The scheme “State Capital Investment Subsidy” ...,20% of Fixed Capital Investment on Plant and m...,A copy of the Memorandum of Association and Ar...,Business & Entrepreneurship,"Powerloom, Incentives, State Capital Investmen...",West Bengal,The scheme “State Capital Investment Subsidy” ...,"[tensor(-0.0261), tensor(0.0404), tensor(-0.00...",S0004


In [ ]:
df=df[df["state"].notnull()]

In [ ]:
df.isnull().sum()

,0
scheme_name,0
details,0
benefits,0
documents,2
schemeCategory,0
tags,0
state,0
tokens,0
embeddings,0
scheme_id,0


In [ ]:
def get_top_k(df, sentence_emb, compare_fn, emb_col="embeddings", k=5,state="Kerala"):
    # Apply the compare function to each embedding in the column
    df_state = df[df['state']==state].copy()
    df_state["similarity"] = df_state[emb_col].apply(lambda emb: compare_fn(sentence_emb, emb))
    top_k = df_state.sort_values("similarity", ascending=False).head(k)
    return top_k

In [ ]:
from flask import Flask, jsonify, request # Import jsonify and request

app = Flask(__name__)

@app.route('/submit', methods=['POST'])
def submit_data():
    try:
        data = request.get_json()
        state=data.get('state')
        prompt = data.get('prompt')
        # Assuming get_schemes is defined elsewhere and works as intended
        schemes_data = get_schemes(prompt,state)
        output = schemes_data[["scheme_id", "scheme_name", "schemeCategory","state"]]
        output = output.to_dict(orient='records')
        return jsonify({"success": True, "message": "SUCCESS", "data": output}), 200
    except Exception as e:
        return jsonify({"success": False, "message": str(e)}), 500

if __name__ == '__main__':
    app.run(debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug: * Restarting with watchdog (inotify)


In [ ]:
get_top_k(df, sentence_emb,compare, emb_col="embeddings", k=5,state="Uttar Pradesh")

,scheme_name,details,benefits,documents,schemeCategory,tags,state,tokens,embeddings,scheme_id,similarity
1799,Mukhyamantri Pragatishil Pashupalak Protsahan ...,"The ""Mukhyamantri Pragatisheel Pashupalak Prot...","Cash awards ranging from ₹10,000 to ₹15,000 pe...",Passport-size Photo. Aadhar card. Certificate ...,"Agriculture,Rural & Environment","Cow, Milk Production, Financial Assistance, Op...",Uttar Pradesh,"The ""Mukhyamantri Pragatisheel Pashupalak Prot...","[tensor(-0.0110), tensor(0.0044), tensor(-0.03...",S1799,0.440569
1819,Mukhyamantri Svadeshi Gau Samvardhan Yojana,"The ""Mukhyamantri Svadeshi Gau Samvardhan Yoja...",Grants covering up to 40% of the total cost pe...,Apply Document: Passport-size Photo. Self-atte...,"Agriculture,Rural & Environment, Women and Chi...","Milk Mission, Production, Financial Assistance...",Uttar Pradesh,"The ""Mukhyamantri Svadeshi Gau Samvardhan Yoja...","[tensor(-0.0144), tensor(-0.0325), tensor(-0.0...",S1819,0.384218
2456,Shravan Kumar Shramik Parivar Tirth Yatra Yojana,The Shravan Kumar Shramik Parivar Tirth Yatra ...,"A maximum lump sum assistance of ₹12,000/- wil...",Attested copy of the online application form f...,"Social welfare & Empowerment, Transport & Infr...","Financial Assistance, Travel, Pilgrimage",Uttar Pradesh,The Shravan Kumar Shramik Parivar Tirth Yatra ...,"[tensor(0.0048), tensor(0.0419), tensor(-0.026...",S2456,0.311304
1630,"Matritva, Shishu Evam Baalika Madad Yojana","The ""Matritva, Shishu Evam Baalika Madad Yojan...",Matritva Hitlabh (Maternity benefits) Register...,Essential Documents: Proof of identity: Attest...,"Women and Child, Social welfare & Empowerment","Financial Assistance, Woman Empowerment, Girl ...",Uttar Pradesh,"The ""Matritva, Shishu Evam Baalika Madad Yojan...","[tensor(-0.0108), tensor(0.0256), tensor(-0.07...",S1630,0.308758
565,Dattopant Thengadi Mratak Shramik Aarthik Saha...,"The ""Dattopant Thengadi Mratak Shramik Aarthik...","Financial Assistance : ₹1,00,000/-",Attested photocopy of the online filled applic...,Social welfare & Empowerment,"Financial Assistance, Social Welfare, Death Be...",Uttar Pradesh,"The ""Dattopant Thengadi Mratak Shramik Aarthik...","[tensor(-0.0480), tensor(0.1176), tensor(-0.12...",S0565,0.286934


In [ ]:
def get_schemes(sentence,state):
    sentence_emb =model.encode([text], convert_to_tensor=True)
    return get_top_k(df, sentence_emb, compare, emb_col="embeddings", k=5,state='x')